# Block-Sparse Featurizer Blocks as Steering and Tuning Targets

A block-sparse featurizer <a id="cite-1"></a>[[1]](#ref-1) encodes an activation into many small blocks of latent
coordinates and keeps only the few blocks of largest norm, so each block names a small subspace rather than one
direction. This example takes two featurizers trained on gemma-3-1b-it's residual stream after block 17, one in the
residual stream's own coordinates and one in J-space, the stream's image under the Jacobian lens
<a id="cite-2"></a>[[2]](#ref-2), and asks what their blocks do on RTE <a id="cite-3"></a>[[3]](#ref-3):

1. **Which blocks carry the answer?** A block's code norm at the answer position, ranked by how well it separates the
   items the model answers Yes from those it answers No.
2. **Do those blocks steer the answer?** Each is ablated, or set to its typical code at a dose, inside the forward
   pass, against random blocks edited the same way.
3. **Can tuning only those blocks move the answer?** The featurizer is spliced in, frozen, and a unit adapter over the
   selected blocks' coordinates is tuned by role through a Fine-Tuning Scheduler schedule, against random blocks.

The blocks are addressed the way any latent model's features are, `features:bsf_resid & layers:17 & units:[b]`, so the
same address selects them for a book, an edit and an adapter.

This notebook needs one GPU with about 20 GB.

In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
N_SCREEN = 96  # validation items the blocks are ranked on
N_EVAL = 96  # held-out validation items every effect is measured on
DOSES = [1.0, 2.0, 4.0]  # multiples of a block's mean code where its gate opens
N_CONTROLS = 16  # random blocks edited the same way
LIMIT_TRAIN_BATCHES = 40  # training batches per epoch, each stage
LIMIT_VAL_BATCHES = 4
ADAPTER_LR = 2.0e-4
SEED = 0
REVISION = (
    None  # component snapshot; None resolves the recorded pin (pass a local-publish rev to validate unmerged configs)
)

In [ ]:
# @title Imports { display-mode: "form" }
import gc
import os
import random

import torch
import yaml
from huggingface_hub import hf_hub_download
from nnsight import TransformersModel

import interpretune as it
from interpretune.adapters.nnsight.backends import NNsightModelBackend
from interpretune.analysis.backends.interventions import PositionScope
from interpretune.analysis.venngram import LatentCatalog
from interpretune.analysis.venngram.latent_edits import block_edit_spec
from interpretune.config import SessionRunnerCfg
from interpretune.runners.core import SessionRunner
from interpretune.tuning.core_loop import FTSCoreCallback
from interpretune.tuning.latents import LatentSite, attach_latent_model
from interpretune.tuning.weights import attach_feature_adapter
from it_examples.seeds import ensure_local_seeds
from it_examples.utils import bsf_featurizers as bf
from it_examples.utils import neuron_finder as nf
from it_examples.utils.nb_ui_utils import display_notebook_table

torch.manual_seed(SEED)
random.seed(SEED)

## 1. The Model and the Behaviour

The session is the `rte_ft.gemma3.core` configuration of the `speediedan/rte` component: gemma-3-1b-it in fp32,
prompted through its chat template to answer whether the premise entails the hypothesis. The behaviour is the answer
**margin**, the Yes logit less the No logit at the answer position. The first `N_SCREEN` validation items rank the
blocks, and the next `N_EVAL` measure every effect.

In [ ]:
# @title 1: The session and the untouched model's answers { display-mode: "form" }
MODEL_ID = "google/gemma-3-1b-it"
ensure_local_seeds()  # idempotent, offline: seed publish sources -> local components cache
if REVISION is None:  # a validation run passes a local-publish REVISION and needs none of these steps
    it.hub.pull("speediedan/rte")  # caches the component's manifest and entrypoint; executes nothing
    os.environ["IT_TRUST_REMOTE_CODE"] = "1"  # opt in to running its entrypoint, for this session only
    it.hub.pull("speediedan/rte", "rte_ft.gemma3.core")


def rte_session(module_cls=None, lr=None):
    "A session on the component's configuration, optionally with a subclassed module and learning rate."
    itdm_cfg, it_cfg, dm_cls, m_cls = it.hub.load("speediedan/rte", "rte_ft.gemma3.core", revision=REVISION)
    if lr is not None:
        it_cfg.optimizer_init["init_args"]["lr"] = lr
    session = it.ITSession(
        it.ITSessionConfig(
            adapter_ctx=(it.Adapter.core,),
            datamodule_cfg=itdm_cfg,
            module_cfg=it_cfg,
            datamodule_cls=dm_cls,
            module_cls=module_cls(m_cls) if module_cls is not None else m_cls,
        )
    )
    it.it_init(**session)
    return session


session = rte_session()
module, tokenizer = session.module, session.datamodule.tokenizer
hf_model = module.model.eval()
hf_model.requires_grad_(False)
device = next(hf_model.parameters()).device
REVISION_OF_MODEL = nf.cached_revision(MODEL_ID)  # the checkpoint the featurizers were trained on
ANSWERS = tuple(int(i) for i in module.it_cfg.entailment_mapping_indices)  # (Yes, No)
N_LAYERS = hf_model.config.num_hidden_layers
batches = list(session.datamodule.val_dataloader())
per_batch = batches[0]["input_ids"].shape[0]
screen_batches = batches[: N_SCREEN // per_batch]
eval_batches = batches[N_SCREEN // per_batch : (N_SCREEN + N_EVAL) // per_batch]
eval_labels = torch.cat([b["labels"].view(-1) for b in eval_batches])


def layer17(model):
    "The decoder layer whose output the featurizers read (block 17)."
    (layer,) = [m for name, m in model.named_modules() if name.endswith("layers.17")]
    return layer


def answers(margins: torch.Tensor) -> dict:
    "Mean margin, Yes rate and accuracy (label 0 is entailment, answered Yes)."
    predicted = (margins <= 0).long()
    return {
        "mean margin": f"{margins.mean():+.2f}",
        "answers Yes": f"{(margins > 0).float().mean():.0%}",
        "accuracy": f"{(predicted == eval_labels).float().mean():.1%}",
    }


@torch.no_grad()
def margins_and_states(model, batches):
    "Each item's answer margin, and block 17's output at the answer position (the last, under left padding)."
    grabbed, margins, states = {}, [], []
    hook = layer17(model).register_forward_hook(lambda m, i, o: grabbed.update(h=(o[0] if isinstance(o, tuple) else o)))
    try:
        for b in batches:
            logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits
            margins.append(nf.answer_margin(logits, ANSWERS).cpu())
            states.append(grabbed["h"][:, -1].float())
    finally:
        hook.remove()
    return torch.cat(margins), torch.cat(states)


base_margins, _ = margins_and_states(hf_model, eval_batches)
screen_margins, screen_states = margins_and_states(hf_model, screen_batches)
base = answers(base_margins)
display_notebook_table(
    [list(base.values())],
    list(base),
    title=f"The untouched model on {len(eval_labels)} held-out RTE items",
    caption=f"{MODEL_ID} at {REVISION_OF_MODEL[:12]}. "
    f"{(eval_labels == 0).float().mean():.0%} of these items are entailed.",
)

## 2. The Featurizers

Both featurizers read block 17's output: one in the residual stream's coordinates (`resid`), one in J-space, the
stream's image $J_{17} h$ under the published Jacobian lens (`jspace`). Each normalizes its input,
$\tilde x = (x - \mu) / s$, encodes 4,096 blocks of three coordinates, $a = \tilde x W_{enc} + b_{enc}$, keeps the 16
blocks of largest norm, and decodes $\hat x = s (a W_{dec}) + \mu$. They are loaded from a Hub repository at a pinned
revision. The fraction of variance their reconstruction leaves unexplained (FVU) at the answer position says how much
of the state the 16 open blocks carry.

In [ ]:
# @title 2: Both featurizers, and their fit at the answer position { display-mode: "form" }
LENS = ("neuronpedia/jacobian-lens", "b25d72a96b79c8e309d6625955a98751da47e67a")
lens_file = "gemma-3-1b-it/jlens/Salesforce-wikitext/gemma-3-1b-it_jacobian_lens.pt"
J = torch.load(hf_hub_download(LENS[0], lens_file, revision=LENS[1]), map_location="cpu", weights_only=True)["J"][17]
J = J.float().to(device)
featurizers = {space: bf.load_featurizer(space, device=device) for space in bf.SPACES}
inputs = {"resid": screen_states, "jspace": screen_states @ J.T}
fit_rows = []
for space, f in featurizers.items():
    x = inputs[space]
    fvu = ((x - f(x)) ** 2).sum() / ((x - x.mean(0)) ** 2).sum()
    fit_rows.append([space, f"{f.n_groups} x {f.group_size}", f.l0, f"{fvu:.3f}"])
display_notebook_table(
    fit_rows,
    ["featurizer", "blocks x coordinates", "open blocks", "FVU at the answer position"],
    title=f"The featurizers at {bf.REPO}@{bf.REVISION[:12]}",
)

## 3. Finding the Blocks That Carry the Answer

A block's code norm at the answer position is a score per item. Its AUROC against "the model answers Yes" says how
well it separates the two answers: near 1, the block is open (or larger) when the model says Yes; near 0, when it
says No. Ranking every block of both featurizers by it finds one decision block toward each answer in each space. The
same ranking against the label (entailed or not) shows how much of the task itself the blocks carry.

In [ ]:
# @title 3: Every block ranked, in both spaces { display-mode: "form" }
def auroc(score: torch.Tensor, positive: torch.Tensor) -> float:
    "The area under the ROC curve of ``score`` for ``positive`` items, by ranks (ties broken by order)."
    ranks = torch.empty_like(score)
    ranks[score.argsort()] = torch.arange(1, len(score) + 1, dtype=score.dtype, device=score.device)
    n1 = positive.sum()
    return float(((ranks * positive).sum() - n1 * (n1 + 1) / 2) / (n1 * (len(positive) - n1)))


says_yes = (screen_margins > 0).float().to(device)
entailed = (torch.cat([b["labels"].view(-1) for b in screen_batches]) == 0).float().to(device)
decision, rank_rows = {}, []
for space, f in featurizers.items():
    norms = f.block_codes(inputs[space]).norm(dim=-1)
    by_answer = torch.tensor([auroc(norms[:, b], says_yes) for b in range(f.n_groups)]).nan_to_num(0.5)
    by_label = torch.tensor([auroc(norms[:, b], entailed) for b in range(f.n_groups)]).nan_to_num(0.5)
    decision[space] = {"yes": int(by_answer.argmax()), "no": int(by_answer.argmin())}
    for answer, b in decision[space].items():
        rank_rows.append(
            [
                space,
                answer,
                f"`features:bsf_{space} & layers:17 & units:[{b}]`",
                f"{by_answer[b]:.3f}",
                f"{(norms[:, b] > 0).float().mean():.0%}",
                f"{float((by_label - 0.5).abs().max()) + 0.5:.3f}",
            ]
        )
display_notebook_table(
    rank_rows,
    ["featurizer", "toward", "address", "AUROC vs answering Yes", "gate open", "best AUROC vs the label"],
    title=f"The decision blocks, on {len(says_yes)} screen items",
)

## 4. The Blocks as a Checkpoint Address Book

A block index means something only for one featurizer trained on one checkpoint, so the book that records the blocks
pins the model by id and revision. Each set names one block of one featurizer; the J-space featurizer's sets carry
`space: jlens`, so a consumer knows their decoder rows are in the lens's coordinates. Any consumer of venngram address
books loads it.

In [ ]:
# @title 4: The book { display-mode: "form" }
book = bf.decision_book(model_revision=REVISION_OF_MODEL, blocks=decision)
print(yaml.safe_dump(book, sort_keys=False))

## 5. Editing a Block Inside the Forward Pass

The residual featurizer is attached to the model backend as a latent model at `blocks.17.hook_resid_post`, with its
error term, so a pass with no edit is the clean pass. `block_edit_spec` compiles an edit of one block's codes into an
op at the featurizer's latent sub-hook, at the answer position:

- `ablate` sets the block's codes to zero;
- `set` writes a dose $d$ times the block's mean code over the screen items where its gate opens.

The decoder is affine, so either edit moves the stream by exactly the block's decoded change, $s (a' - a) D_b$.

**Control (parity):** the same edit written by hand as a forward hook on block 17 must give the same margins.

**Control (random blocks):** `N_CONTROLS` blocks open on at least 10% of the screen items, edited the same way. A
uniformly random block is almost never open at the answer position, so it would be a weak control.

In [ ]:
# @title 5: The decision blocks and random blocks, ablated and set { display-mode: "form" }
torch.set_grad_enabled(False)
bsf = featurizers["resid"]
catalog = LatentCatalog([bsf.binding("bsf_resid")])
lm = TransformersModel(hf_model, tokenizer=tokenizer)  # nnsight, around the same weights
backend = NNsightModelBackend.for_model(lm)
screen_codes = bsf.block_codes(screen_states)
open_rate = (screen_codes.norm(dim=-1) > 0).float().mean(0).cpu()


def mean_code(b: int) -> torch.Tensor:
    "Block ``b``'s mean code over the screen items where its gate opens."
    c = screen_codes[:, b]
    return c[c.norm(dim=-1) > 0].mean(0).cpu()


def edit(b: int, op: str, dose: float | None = None):
    target = catalog.resolve(f"features:bsf_resid & layers:17 & units:[{b}]", n_layers=N_LAYERS).targets[0]
    value = None if op == "ablate" else dose * mean_code(b)
    return block_edit_spec(catalog, target, op, value=value, position_scope=PositionScope.LAST_TOKEN)


def natively(hook_and_spec) -> torch.Tensor:
    "The held-out margins with one compiled edit, the featurizer attached."
    hook, spec = hook_and_spec
    edited = []
    for b in eval_batches:
        inputs = {"input_ids": b["input_ids"].to(device), "attention_mask": b["attention_mask"].to(device)}
        _, logits = backend.fwd_w_intervention(lm, inputs, {hook: spec}, latent_model_handles=[bsf])
        edited.append(nf.answer_margin(logits, ANSWERS).cpu())
    return torch.cat(edited)


def by_hand(b: int, op: str, dose: float | None = None) -> torch.Tensor:
    "The same edit as a forward hook on block 17: new codes for block b at the answer position, decoded back."

    def hook(_m, _i, out):
        h = out[0] if isinstance(out, tuple) else out
        last = h[:, -1].float()
        codes = bsf.block_codes(last)[:, b]
        new = torch.zeros_like(codes) if op == "ablate" else (dose * mean_code(b)).to(codes).expand_as(codes)
        h = h.clone()
        h[:, -1] = (last + ((new - codes) @ bsf.block(b)) * bsf.scale).to(h.dtype)
        return (h, *out[1:]) if isinstance(out, tuple) else h

    handle = layer17(hf_model).register_forward_hook(hook)
    try:
        return margins_and_states(hf_model, eval_batches)[0]
    finally:
        handle.remove()


EDITS = [("ablate", None)] + [("set", d) for d in DOSES]
shift = lambda m: float((m - base_margins).mean())  # noqa: E731
edit_rows, parity = [], 0.0
for answer, b in decision["resid"].items():
    for op, dose in EDITS:
        m = natively(edit(b, op, dose))
        if op == "ablate" or dose == max(DOSES):
            parity = max(parity, float((m - by_hand(b, op, dose)).abs().max()))
        edit_rows.append(
            [f"{answer} block {b}", op if dose is None else f"{op} x{dose:g}", f"{shift(m):+.2f}", *answers(m).values()]
        )
pool = [b for b in (open_rate >= 0.1).nonzero().flatten().tolist() if b not in decision["resid"].values()]
controls = random.Random(SEED).sample(pool, min(N_CONTROLS, len(pool)))
control_shifts = {
    (op, dose): torch.tensor([shift(natively(edit(b, op, dose))) for b in controls])
    for op, dose in (("ablate", None), ("set", max(DOSES)))
}
display_notebook_table(
    edit_rows,
    ["block", "edit", "margin shift", *base],
    title="The decision blocks, edited natively at the answer position",
    caption=f"Untouched: {base['mean margin']} margin, {base['answers Yes']} Yes, {base['accuracy']} accuracy. "
    f"Largest margin difference from the same edit by hand: {parity:.4f}.",
)
display_notebook_table(
    [
        [
            op if dose is None else f"{op} x{dose:g}",
            f"{v.abs().mean():.2f}",
            f"{v.abs().quantile(0.95):.2f}",
            f"{v.abs().max():.2f}",
        ]
        for (op, dose), v in control_shifts.items()
    ],
    ["edit", "mean |shift|", "p95 |shift|", "max |shift|"],
    title=f"{len(controls)} random blocks open on at least 10% of the screen items, edited the same way",
)
assert parity < 0.05, f"the native edit is not the edit written by hand: {parity}"

## 6. Tuning Only the Decision Blocks

Editing sets a block's codes by hand. Tuning learns a change to the featurizer instead, from labelled data, at the
selected blocks only. The featurizer is spliced in at block 17 as a frozen latent model (`attach_latent_model`, role
`artifact`) through `KeptError`, which writes $x + f_\theta(x) - f_0(x)$: the stream plus the change the tuned
featurizer $f_\theta$ makes to the loaded one's reconstruction $f_0$. With nothing tuned the model is unchanged.

A **feature adapter** (`attach_feature_adapter`) is a trainable delta over the coordinates of the blocks an address
selects, on each of the featurizer's tensors with a latent axis: block $b$'s three encoder columns and bias entries,
and its three decoder rows. Two adapters, `to_no` over the decision block toward No and `to_yes` over the one toward
Yes, take the roles `adapter:to_no` and `adapter:to_yes`, and a Fine-Tuning Scheduler schedule thaws them by role, in
two experiment stages:

```python
{0: {"params": ["role:adapter:to_no"], "max_transition_epoch": 1}, 1: {"params": ["role:adapter:to_yes"]}}
```

Every other tensor stays frozen, the model's and the featurizer's own. The objective is RTE's labels on the training
split.

**The control** is the same run on two random blocks open on at least 10% of the screen items.

In [ ]:
# @title 6: Two tuning runs, the decision blocks and random ones { display-mode: "form" }
# free the first session (its model, the nnsight wrapper and the attached featurizer) before a tuning session loads its
# own copy of the model
lm = backend = module = hf_model = session = catalog = bsf = None
featurizers.clear()
gc.collect()
torch.cuda.empty_cache()
torch.set_grad_enabled(True)


def with_block_adapters(adapters: dict[str, list[int]]):
    "A module class that splices the residual featurizer in and attaches one feature adapter per name."

    def subclass(base_cls):
        class BlockAdapted(base_cls):
            def register_tunable_modules(self) -> None:
                super().register_tunable_modules()
                kept = bf.KeptError(bf.load_featurizer("resid", device=next(self.model.parameters()).device))
                site = LatentSite("model.model.layers.17", op="replace")
                attach_latent_model(self, "bsf_resid", kept, site, frozen=True)
                adapters_catalog = LatentCatalog([kept.binding("bsf_resid")])
                for name, blocks in adapters.items():
                    address = f"features:bsf_resid & layers:17 & units:{blocks}"
                    attach_feature_adapter(self, adapters_catalog, address, name=name, n_layers=N_LAYERS)

        return BlockAdapted

    return subclass


def tune(adapters: dict[str, list[int]]):
    "Tune ``adapters`` with the two-stage schedule; returns the held-out margins before and after, and the count."
    torch.manual_seed(SEED)
    run = rte_session(with_block_adapters(adapters), lr=ADAPTER_LR)
    roles = set(run.module.tunable_view.roles().values())
    assert {"adapter:to_no", "adapter:to_yes", "artifact"} <= roles, roles
    model = run.module.model.eval()
    before = margins_and_states(model, eval_batches)[0]
    schedule = {
        0: {"params": ["role:adapter:to_no"], "max_transition_epoch": 1},
        1: {"params": ["role:adapter:to_yes"]},
    }
    fts = FTSCoreCallback(schedule)
    SessionRunner(
        run_cfg=SessionRunnerCfg(
            it_session=run,
            max_epochs=2,
            limit_train_batches=LIMIT_TRAIN_BATCHES,
            limit_val_batches=LIMIT_VAL_BATCHES,
            callbacks=[fts],
        )
    ).train()
    assert fts.tuning_scheduler.curr_depth == 1, "both stages must run"
    trained = sum(p.numel() for p in run.module.parameters() if p.requires_grad)
    after = margins_and_states(run.module.model.eval(), eval_batches)[0]
    del run, model
    torch.cuda.empty_cache()
    return before, after, trained


found_adapters = {"to_no": [decision["resid"]["no"]], "to_yes": [decision["resid"]["yes"]]}
random_blocks = random.Random(SEED + 1).sample(pool, 2)
random_adapters = {"to_no": [random_blocks[0]], "to_yes": [random_blocks[1]]}
spliced_found, tuned_found, n_found = tune(found_adapters)
spliced_random, tuned_random, n_random = tune(random_adapters)
assert torch.allclose(spliced_found, base_margins, atol=1e-3), "the untuned splice must leave the model unchanged"
display_notebook_table(
    [
        ["untouched", "", *base.values()],
        [f"decision blocks {found_adapters}", f"{n_found:,}", *answers(tuned_found).values()],
        [f"random blocks {random_adapters}", f"{n_random:,}", *answers(tuned_random).values()],
    ],
    ["tuned", "trainable parameters", *base],
    title="Tuning two blocks of the frozen featurizer, by role",
)

## What This Showed

On 96 held-out RTE items, gemma-3-1b-it with this prompt answers Yes on 40% of them, at 77.1% accuracy.

- **The answer has a block in each featurizer.** Ranked by the AUROC of their code norm against "the model answers
  Yes", one block toward each answer stands out in each space: residual blocks 327 (0.993) and 3504 (0.016), J-space
  blocks 1048 (0.994) and 4068 (0.003). Ranked on this session's own items, they are the blocks the featurizers'
  authors found on a separate evaluation set (the J-space No block was their second). The label is carried much less
  well: no block separates entailed from not entailed items better than an AUROC of about 0.78. These blocks track
  what the model will say, not whether it is right.
- **The native edit is the edit.** Each block edit compiled by `block_edit_spec` and run at the featurizer's latent
  sub-hook moves the margins exactly as the same edit written by hand as a forward hook does (largest difference
  0.0001), and a pass with the featurizer attached and no edit is the clean pass.
- **Setting the No block writes No; the other edits are within what random blocks do.** Setting residual block 3504
  to four times its typical code takes the mean margin from -3.79 to -14.21 and the Yes rate to 0%, a shift of
  -10.43, larger than any of 14 random blocks (each open on at least 10% of the screen items) edited the same way (largest 8.05).
  Setting block 327 moves the margin toward Yes, monotonically with the dose (+0.29, +2.32, +6.44), and ablating
  either block moves it the other way (-1.98 and +3.33), each in the block's own direction but within the random
  blocks' range: such a random block, set to four times its typical code, shifts the margin by
  2.08 on average and by 6.66 at the 95th percentile. So the ranking finds blocks that move the answer in their
  direction, but only one of them moves it further than a random open block can.
- **A frozen featurizer tunes at two blocks, and the blocks matter little for accuracy.** With nothing trained, the
  spliced featurizer leaves every margin as it was. Tuning the two decision blocks (13,830 parameters, two stages by
  role) moves the answers toward Yes (mean margin +2.11, Yes on 56% of items) at the same accuracy, 77.1%. Tuning two
  random blocks moves them further toward Yes (75%) and costs accuracy (72.9%). On this budget neither run learns the
  task better than the untouched model: the adapters shift the decision threshold, and the decision blocks are the
  ones that shift it without losing accuracy.

**What this example does not cover.** The edits run on nnsight: Interpretune's TransformerLens backend attaches latent
models through sae_lens' `SAETransformerBridge`, which takes sae_lens SAEs, so a featurizer that satisfies the
latent-model protocol attaches here but not there. The J-space featurizer's blocks are ranked and recorded, not edited:
an edit in J-space needs the lens's inverse to reach the residual stream. Comparing the blocks with transcoder
features at the same layer is open.


## References

1. <a id="ref-1"></a>Goodfire. "block-sparse-featurizer." 2026.
   [github.com/goodfire-ai/block-sparse-featurizer](https://github.com/goodfire-ai/block-sparse-featurizer).
   [↩](#cite-1)
2. <a id="ref-2"></a>Neuronpedia. "jacobian-lens." 2026.
   [huggingface.co/neuronpedia/jacobian-lens](https://huggingface.co/neuronpedia/jacobian-lens). [↩](#cite-2)
3. <a id="ref-3"></a>Alex Wang et al. "SuperGLUE: A Stickier Benchmark for General-Purpose Language Understanding
   Systems." NeurIPS 2019. arXiv:1905.00537. [arxiv.org/abs/1905.00537](https://arxiv.org/abs/1905.00537).
   [↩](#cite-3)